In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
sns.set_theme(style="whitegrid", font_scale=1.0)

In [3]:
df = pd.read_csv('Car_Price_Cleaned.csv')

Optimal Alpha (penalty): 0.3511
Test R^2 Score: 0.8806
Test RMSE (Log Scale): 0.1800


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

# 1. Feature Set & Target Definition
num_features = ['curbweight', 'enginesize', 'horsepower', 'citympg']
cat_features = ['drivewheel', 'carbody', 'enginelocation']

X = df[num_features + cat_features]
y_log = np.log1p(df['price'])  # Log transform price to handle skewness
y_actual = df['price']

# 2. Train-Test Split (Splitting both log target and actual dollar target)
X_train, X_test, y_train_log, y_test_log, y_train_act, y_test_act = train_test_split(
    X, y_log, y_actual, test_size=0.2, random_state=42
)

# 3. Preprocessing Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first'), cat_features)
    ]
)

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

# 4. Fit RidgeCV Model
alphas = np.logspace(-3, 3, 100)
ridge_model = RidgeCV(alphas=alphas)
ridge_model.fit(X_train_prep, y_train_log)

# 5. Predict on Test Set
y_pred_log = ridge_model.predict(X_test_prep)
y_pred_dollar = np.expm1(y_pred_log)  # Exponentiate back to actual dollar values

# 6. Calculate Detailed Performance Metrics
n = X_test.shape[0]          # Number of test observations
p = X_train_prep.shape[1]    # Number of features after One-Hot Encoding

In [7]:
# Metrics on Original Dollar Scale
r2 = r2_score(y_test_act, y_pred_dollar)
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
mae = mean_absolute_error(y_test_act, y_pred_dollar)
rmse = np.sqrt(mean_squared_error(y_test_act, y_pred_dollar))
mape = mean_absolute_percentage_error(y_test_act, y_pred_dollar) * 100

# 7. Display Results
print("=" * 45)
print("       RIDGE REGRESSION MODEL PERFORMANCE     ")
print("=" * 45)
print(f"Optimal Alpha Penalty : {ridge_model.alpha_:.4f}")
print("-" * 45)
print(f"R² Score              : {r2:.4f}")
print(f"Adjusted R² Score     : {adj_r2:.4f}")
print(f"Mean Absolute Error   : ${mae:,.2f}")
print(f"Root Mean Sq. Error   : ${rmse:,.2f}")
print(f"Mean Abs. % Error     : {mape:.2f}%")
print("=" * 45)

       RIDGE REGRESSION MODEL PERFORMANCE     
Optimal Alpha Penalty : 0.3511
---------------------------------------------
R² Score              : 0.8576
Adjusted R² Score     : 0.8036
Mean Absolute Error   : $2,054.84
Root Mean Sq. Error   : $3,352.62
Mean Abs. % Error     : 13.64%


In [15]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [8]:
df_clean = df.copy()

In [11]:
# Define Feature Sets
num_cols = ['enginesize', 'horsepower', 'curbweight', 'carwidth', 'carlength', 'wheelbase', 'boreratio', 'citympg', 'highwaympg', 'cylindernumber', 'doornumber']
cat_cols = ['manufacturer', 'carbody', 'drivewheel', 'enginelocation', 'fueltype', 'aspiration', 'fuelsystem']

In [12]:
X = df_clean[num_cols + cat_cols]
y = np.log(df_clean['price'])  # Log-transform target to handle skewness

In [13]:
# Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
# -------------------------------------------------------------------
# 2. Define Shared Preprocessing Transformers
# -------------------------------------------------------------------
preprocessor_standard = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
])

preprocessor_pca = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=0.90))  # Retain 90% of numerical feature variance
    ]), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
])

In [17]:
# -------------------------------------------------------------------
# 3. Build Comparison Pipelines
# -------------------------------------------------------------------
pipelines = {
    'OLS (Baseline)': Pipeline([('prep', preprocessor_standard), ('reg', LinearRegression())]),
    'PCR (PCA + OLS)': Pipeline([('prep', preprocessor_pca), ('reg', LinearRegression())]),
    'Ridge (L2 Penalty)': Pipeline([('prep', preprocessor_standard), ('reg', Ridge(alpha=1.0))])
}

In [18]:
# -------------------------------------------------------------------
# 4. Cross-Validation & Test Set Evaluation Loop
# -------------------------------------------------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

print("=== 5-FOLD CROSS-VALIDATION RESULTS (LOG SCALE R²) ===")
for name, pipe in pipelines.items():
    # 5-Fold Cross Validation on Training Data
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=kf, scoring='r2')
    
    # Fit model on training set and evaluate on test set
    pipe.fit(X_train, y_train)
    y_pred_log = pipe.predict(X_test)
    
    # Convert predictions back to original Dollar Values ($)
    y_test_dollars = np.exp(y_test)
    y_pred_dollars = np.exp(y_pred_log)
    
    # Compute metrics in original currency
    test_r2 = r2_score(y_test_dollars, y_pred_dollars)
    test_mae = mean_absolute_error(y_test_dollars, y_pred_dollars)
    test_rmse = np.sqrt(mean_squared_error(y_test_dollars, y_pred_dollars))
    
    results.append({
        'Model Pipeline': name,
        'Mean CV R²': f"{cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})",
        'Test R² ($)': round(test_r2, 4),
        'Test MAE ($)': f"${test_mae:,.2f}",
        'Test RMSE ($)': f"${test_rmse:,.2f}"
    })

=== 5-FOLD CROSS-VALIDATION RESULTS (LOG SCALE R²) ===


/Users/mariaoanacucu/Documents/VS Code Projects/Car_feature_price_analysis/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:228: UserWarning: Found unknown categories in columns [0, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/mariaoanacucu/Documents/VS Code Projects/Car_feature_price_analysis/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:228: UserWarning: Found unknown categories in columns [6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/mariaoanacucu/Documents/VS Code Projects/Car_feature_price_analysis/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:228: UserWarning: Found unknown categories in columns [0, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/mariaoanacucu/Documents/VS Code Projects/Car_feature_price_analysis/.venv/lib/python3.12/site-packages/sklearn/

In [19]:
# Output Summary Comparison Table
results_df = pd.DataFrame(results)
results_df

,Model Pipeline,Mean CV R²,Test R² ($),Test MAE ($),Test RMSE ($)
0,OLS (Baseline),0.8882 (+/- 0.0270),0.9277,"$1,634.30","$2,388.36"
1,PCR (PCA + OLS),0.9083 (+/- 0.0269),0.9098,"$1,683.89","$2,669.15"
2,Ridge (L2 Penalty),0.9070 (+/- 0.0269),0.9421,"$1,470.40","$2,137.85"
